In [2]:
import os
os.getcwd()

'd:\\Programming\\labwork\\ml'

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import sys

np.random.seed(0)

def load_csv(filename):
    df = pd.read_csv(filename)
    if {'x1', 'x2', 'y'}.issubset(df.columns):
        X = df[['x1', 'x2']].values.astype(float)
        y = df['y'].values.astype(float)
    else:
        X = df.iloc[:, :2].values.astype(float)
        y = df.iloc[:, 2].values.astype(float)
    y = np.where(y <= 0, -1.0, 1.0)
    return X, y

class LinearSVM_SGD:
    def __init__(self, C=1.0, lr=0.01, epochs=200, verbose=False):
        self.C = float(C)
        self.lr = float(lr)
        self.epochs = int(epochs)
        self.verbose = verbose
        self.w = None
        self.b = 0.0

    def fit(self, X, y):
        n, d = X.shape
        self.w = np.zeros(d, dtype=float)
        self.b = 0.0
        for epoch in range(self.epochs):
            idx = np.random.permutation(n)
            Xs = X[idx]
            ys = y[idx]
            for i in range(n):
                xi = Xs[i]
                yi = ys[i]
                margin = yi * (np.dot(self.w, xi) + self.b)
                if margin < 1:
                    grad_w = self.w - self.C * yi * xi
                    grad_b = - self.C * yi
                else:
                    grad_w = self.w
                    grad_b = 0.0
                lr_t = self.lr / (1.0 + epoch * 1e-3)
                self.w = self.w - lr_t * grad_w
                self.b = self.b - lr_t * grad_b
            if self.verbose and (epoch % (max(1, self.epochs // 5)) == 0):
                pred = self.predict(X)
                acc = np.mean(pred == y)
                print(f"Epoch {epoch}/{self.epochs} val acc {acc:.4f}")
        return self

    def decision_function(self, X):
        return X.dot(self.w) + self.b

    def predict(self, X):
        scores = self.decision_function(X)
        return np.where(scores >= 0, 1.0, -1.0)

def plot_linear_decision_boundary(model, X, y, filename='linear_decision_boundary.png', title='Linear SVM decision boundary'):
    x_min, x_max = X[:,0].min() - 0.5, X[:,0].max() + 0.5
    y_min, y_max = X[:,1].min() - 0.5, X[:,1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400), np.linspace(y_min, y_max, 400))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure(figsize=(8,6))
    plt.contourf(xx, yy, np.sign(Z), alpha=0.1, levels=[-1,0,1], cmap='coolwarm')
    cs = plt.contour(xx, yy, Z, levels=[-1.0, 0.0, 1.0], linestyles=['--', '-', '--'])
    plt.clabel(cs, inline=1, fontsize=10)
    plt.scatter(X[y==1,0], X[y==1,1], marker='o', edgecolor='k', label='+1')
    plt.scatter(X[y==-1,0], X[y==-1,1], marker='s', edgecolor='k', label='-1')
    plt.legend()
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved linear boundary plot to {filename}")

def rbf_kernel(X1, X2, gamma):
    X1s = np.sum(X1**2, axis=1)[:, None]
    X2s = np.sum(X2**2, axis=1)[None, :]
    K = np.exp(-gamma * (X1s + X2s - 2.0 * np.dot(X1, X2.T)))
    return K

def poly_kernel(X1, X2, degree=3, coef0=1.0):
    return (np.dot(X1, X2.T) + coef0) ** degree

class KernelSVM_SMO:
    def __init__(self, C=1.0, kernel='rbf', kernel_params=None, tol=1e-3, max_passes=20, verbose=False):
        self.C = float(C)
        self.kernel = kernel
        self.kernel_params = kernel_params or {}
        self.tol = tol
        self.max_passes = max_passes
        self.verbose = verbose
        self.X = None
        self.y = None
        self.alpha = None
        self.b = 0.0
        self.K = None

    def _kernel_matrix(self, X):
        if self.kernel == 'rbf':
            gamma = self.kernel_params.get('gamma', 1.0)
            return rbf_kernel(X, X, gamma)
        elif self.kernel == 'poly':
            deg = self.kernel_params.get('degree', 3)
            c0 = self.kernel_params.get('coef0', 1.0)
            return poly_kernel(X, X, degree=deg, coef0=c0)
        else:
            raise ValueError("Unknown kernel")

    def fit(self, X, y):
        n = X.shape[0]
        self.X = X.copy()
        self.y = y.copy()
        self.alpha = np.zeros(n, dtype=float)
        self.b = 0.0
        self.K = self._kernel_matrix(self.X)
        passes = 0
        it = 0
        while passes < self.max_passes:
            num_changed_alphas = 0
            for i in range(n):
                f_i = np.sum(self.alpha * self.y * self.K[:, i]) + self.b
                E_i = f_i - self.y[i]
                if (self.y[i] * E_i < -self.tol and self.alpha[i] < self.C) or (self.y[i] * E_i > self.tol and self.alpha[i] > 0):
                    j = np.random.randint(0, n-1)
                    if j >= i:
                        j += 1
                    f_j = np.sum(self.alpha * self.y * self.K[:, j]) + self.b
                    E_j = f_j - self.y[j]
                    alpha_i_old = self.alpha[i].copy()
                    alpha_j_old = self.alpha[j].copy()
                    if self.y[i] != self.y[j]:
                        L = max(0.0, self.alpha[j] - self.alpha[i])
                        H = min(self.C, self.C + self.alpha[j] - self.alpha[i])
                    else:
                        L = max(0.0, self.alpha[i] + self.alpha[j] - self.C)
                        H = min(self.C, self.alpha[i] + self.alpha[j])
                    if L == H:
                        continue
                    eta = 2.0 * self.K[i, j] - self.K[i, i] - self.K[j, j]
                    if eta >= 0:
                        continue
                    self.alpha[j] = self.alpha[j] - (self.y[j] * (E_i - E_j)) / eta
                    if self.alpha[j] > H:
                        self.alpha[j] = H
                    elif self.alpha[j] < L:
                        self.alpha[j] = L
                    if abs(self.alpha[j] - alpha_j_old) < 1e-8:
                        continue
                    self.alpha[i] = self.alpha[i] + self.y[i] * self.y[j] * (alpha_j_old - self.alpha[j])
                    b1 = self.b - E_i - self.y[i] * (self.alpha[i] - alpha_i_old) * self.K[i, i] - self.y[j] * (self.alpha[j] - alpha_j_old) * self.K[i, j]
                    b2 = self.b - E_j - self.y[i] * (self.alpha[i] - alpha_i_old) * self.K[i, j] - self.y[j] * (self.alpha[j] - alpha_j_old) * self.K[j, j]
                    if 0 < self.alpha[i] < self.C:
                        self.b = b1
                    elif 0 < self.alpha[j] < self.C:
                        self.b = b2
                    else:
                        self.b = 0.5 * (b1 + b2)
                    num_changed_alphas += 1
            if self.verbose:
                print(f"SMO pass {passes}: changed {num_changed_alphas} alphas")
            if num_changed_alphas == 0:
                passes += 1
            else:
                passes = 0
            it += 1
            if it > 10000:
                if self.verbose:
                    print("Stopping early due to iteration cap")
                break
        return self

    def project(self, X_test):
        if self.kernel == 'rbf':
            gamma = self.kernel_params.get('gamma', 1.0)
            K_test = rbf_kernel(self.X, X_test, gamma)
        elif self.kernel == 'poly':
            deg = self.kernel_params.get('degree', 3)
            c0 = self.kernel_params.get('coef0', 1.0)
            K_test = poly_kernel(self.X, X_test, degree=deg, coef0=c0)
        else:
            raise ValueError("Unknown kernel")
        return np.dot((self.alpha * self.y), K_test) + self.b

    def predict(self, X_test):
        return np.where(self.project(X_test) >= 0, 1.0, -1.0)

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def main():
    try:
        X_train, y_train = load_csv('svm_train_data.csv')
        X_val, y_val = load_csv('svm_val_data.csv')
        X_test, y_test = load_csv('svm_test_data.csv')
    except Exception as e:
        print("Error reading CSVs. Make sure files exist in current dir and columns are x1,x2,y or first two features and last column label.")
        print("Exception:", e)
        sys.exit(1)

    print(f"Loaded: train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")

    Cs = [0.01, 0.1, 1, 10, 100]
    best_C = None
    best_val_acc = -1.0
    results = {}
    print("\nTraining Linear SVM (SGD) over C values:", Cs)
    for C in Cs:
        lr = 0.01
        svm = LinearSVM_SGD(C=C, lr=lr, epochs=400, verbose=False)
        svm.fit(X_train, y_train)
        val_pred = svm.predict(X_val)
        val_acc = accuracy(y_val, val_pred)
        test_pred = svm.predict(X_test)
        test_acc = accuracy(y_test, test_pred)
        results[C] = (val_acc, test_acc, svm)
        print(f"C={C}: val acc={val_acc*100:.2f}%, test acc={test_acc*100:.2f}%")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_C = C
    print(f"Best linear C by validation: {best_C} (val acc {best_val_acc*100:.2f}%)")

    best_model = results[best_C][2]
    best_val_acc = results[best_C][0]
    best_test_acc = results[best_C][1]

    plot_linear_decision_boundary(best_model, X_train, y_train, filename='linear_decision_boundary.png', title=f'Linear SVM (C={best_C})')

    print("\nLinear SVM results (rounded to 2 decimals):")
    print(f"Validation accuracy: {best_val_acc*100:.2f}%")
    print(f"Test accuracy:       {best_test_acc*100:.2f}%")

    print("\nTraining Kernel SVM (Simplified SMO) with RBF kernel")
    gammas = [0.5, 1.0, 2.0]
    C_kernel = 1.0
    best_g = None
    best_val_kacc = -1.0
    best_kmodel = None
    for g in gammas:
        ks = KernelSVM_SMO(C=C_kernel, kernel='rbf', kernel_params={'gamma': g}, tol=1e-3, max_passes=30, verbose=False)
        start = time.time()
        ks.fit(X_train, y_train)
        t = time.time() - start
        val_pred = ks.predict(X_val)
        val_acc = accuracy(y_val, val_pred)
        test_acc = accuracy(y_test, ks.predict(X_test))
        print(f"gamma={g}: val acc={val_acc*100:.2f}%, test acc={test_acc*100:.2f}%, time {t:.2f}s")
        if val_acc > best_val_kacc:
            best_val_kacc = val_acc
            best_g = g
            best_kmodel = ks
    print(f"Best RBF gamma by validation: {best_g} (val acc {best_val_kacc*100:.2f}%)")
    print("\nKernel SVM (RBF) results (rounded to 2 decimals):")
    print(f"Validation accuracy: {best_val_kacc*100:.2f}%")
    print(f"Test accuracy:       {accuracy(y_test, best_kmodel.predict(X_test))*100:.2f}%")

    print("\nSUMMARY (rounded to 2 decimals):")
    print(f"Linear SVM (best C={best_C})  -> val: {best_val_acc*100:.2f}%, test: {best_test_acc*100:.2f}%")
    print(f"Kernel SVM (RBF best gamma={best_g}, C={C_kernel}) -> val: {best_val_kacc*100:.2f}%, test: {accuracy(y_test, best_kmodel.predict(X_test))*100:.2f}%")
    print("\nDone.")

if __name__ == '__main__':
    main()


Loaded: train (180, 2), val (60, 2), test (60, 2)

Training Linear SVM (SGD) over C values: [0.01, 0.1, 1, 10, 100]
C=0.01: val acc=50.00%, test acc=41.67%
C=0.1: val acc=50.00%, test acc=41.67%
C=1: val acc=70.00%, test acc=78.33%
C=10: val acc=76.67%, test acc=83.33%
C=100: val acc=75.00%, test acc=86.67%
Best linear C by validation: 10 (val acc 76.67%)
Saved linear boundary plot to linear_decision_boundary.png

Linear SVM results (rounded to 2 decimals):
Validation accuracy: 76.67%
Test accuracy:       83.33%

Training Kernel SVM (Simplified SMO) with RBF kernel
gamma=0.5: val acc=88.33%, test acc=98.33%, time 0.82s
gamma=1.0: val acc=95.00%, test acc=98.33%, time 2.79s
gamma=2.0: val acc=95.00%, test acc=98.33%, time 0.68s
Best RBF gamma by validation: 1.0 (val acc 95.00%)

Kernel SVM (RBF) results (rounded to 2 decimals):
Validation accuracy: 95.00%
Test accuracy:       98.33%

SUMMARY (rounded to 2 decimals):
Linear SVM (best C=10)  -> val: 76.67%, test: 83.33%
Kernel SVM (RBF be